In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

## 1. PINN Architecture Definition
Definition of the standard Neural Network to approximate the solution $u(t,x)$ containing the trainable parameter `alpha`.

In [ ]:
class PinnHeatEq(nn.Module):
    def __init__(self):
        super().__init__()
        self.couche_entree = nn.Linear(2, 50)
        self.couche_cachee1 = nn.Linear(50, 50)
        self.couche_cachee2 = nn.Linear(50, 50)
        self.couche_sortie = nn.Linear(50, 1)
        # Initialising alpha by 0.01 randomly
        self.alpha = nn.Parameter(torch.tensor([0.01], dtype=torch.float32))

    def forward(self, x):
        x = torch.tanh(self.couche_entree(x))
        x = torch.tanh(self.couche_cachee1(x))
        x = torch.tanh(self.couche_cachee2(x))
        return self.couche_sortie(x)

## 2. Sampling, Collocation, and Observation Points Generation
Functions to generate collocation points (in the domain), initial points ($t=0$), boundary points ($x=0$ and $x=1$), and synthetic observation data used for identifying $\alpha$.

In [ ]:
t_min, t_max = 0.0, 1.0
x_min, x_max = 0.0, 1.0


def generer_points_collocation(n_pde):
    t_colloc = torch.rand(n_pde, 1) * (t_max - t_min) + t_min
    x_colloc = torch.rand(n_pde, 1) * (x_max - x_min) + x_min
    return t_colloc.float(), x_colloc.float()


def generer_points_initiaux(n_iv):
    t_init = torch.zeros(n_iv, 1)
    x_init = torch.rand(n_iv, 1) * (x_max - x_min) + x_min
    return t_init.float(), x_init.float()


def generer_points_bords(n_bords):
    t_bord = torch.rand(n_bords, 1) * (t_max - t_min) + t_min
    x_gauche = torch.ones(n_bords // 2, 1) * x_min
    x_droite = torch.ones(n_bords // 2, 1) * x_max
    x_bord = torch.cat([x_gauche, x_droite], dim=0)
    return t_bord.float(), x_bord.float()


# Génération des données synthétiques d'observation
t_data = torch.rand(400, 1)
x_data = torch.rand(400, 1) * (x_max - x_min) + x_min
inputs_data = torch.cat([t_data, x_data], dim=1)
alpha_vrai = 0.0005
u_vrai = torch.exp(-alpha_vrai * 4 * (torch.pi**2) * t_data) * torch.sin(2 * torch.pi * x_data)
u_data = u_vrai

## 3. Loss Functions Definition
Grouping of loss calculation functions for initial conditions (IV), boundary conditions (BC), PDE residual (CLP), and observation data.

In [ ]:
def calc_iv_loss(model, t_init, x_init, u_exact_init):
    u_pred = model(torch.cat([t_init, x_init], dim=1))
    return torch.mean((u_pred - u_exact_init) ** 2)


def calc_bc_loss(model, t_bord, x_bord, u_exact_bord):
    u_pred = model(torch.cat([t_bord, x_bord], dim=1))
    return torch.mean((u_pred - u_exact_bord) ** 2)


def calc_clp_loss(model, t_colloc, x_colloc, alpha):
    t_colloc.requires_grad_(True)
    x_colloc.requires_grad_(True)

    u_pred = model(torch.cat([t_colloc, x_colloc], dim=1))

    u_t = torch.autograd.grad(
        outputs=u_pred,
        inputs=t_colloc,
        grad_outputs=torch.ones_like(u_pred),
        create_graph=True,
    )[0]

    u_x = torch.autograd.grad(
        outputs=u_pred,
        inputs=x_colloc,
        grad_outputs=torch.ones_like(u_pred),
        create_graph=True,
    )[0]

    u_xx = torch.autograd.grad(
        outputs=u_x,
        inputs=x_colloc,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True,
    )[0]

    residu = u_t - alpha * u_xx
    return torch.mean(residu ** 2)


def calc_data_loss(model, inputs_data, u_data):
    u_pred_data = model(inputs_data)
    return torch.mean((u_pred_data - u_data) ** 2)

## 4. Hardware (Device), Model, and Data Initialization
Hardware detection, inverse model creation, definition of the Adam optimizer with two distinct learning rates (parameter group for `alpha`), and transfer of points to the device.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

modele = PinnHeatEq().to(device)


optimizer = optim.Adam([
    {'params': [p for n, p in modele.named_parameters() if n != 'alpha'], 'lr': 0.001},
    {'params': [modele.alpha], 'lr': 0.02} # different lr to get a fast convergence
])

t_colloc, x_colloc = generer_points_collocation(12000)
t_init, x_init = generer_points_initiaux(1000)
t_bord, x_bord = generer_points_bords(1000)

u_exact_init = torch.sin(2 * torch.pi * x_init)
u_exact_bord = torch.zeros_like(x_bord)

t_colloc, x_colloc = t_colloc.to(device), x_colloc.to(device)
t_init, x_init = t_init.to(device), x_init.to(device)
t_bord, x_bord = t_bord.to(device), x_bord.to(device)
u_exact_init = u_exact_init.to(device)
u_exact_bord = u_exact_bord.to(device)

inputs_data = inputs_data.to(device)
u_data = u_data.to(device)

## 5. Model Training (Adam)
Training phase of the inverse problem with the Adam optimizer to identify the physical parameter $\alpha$.

In [ ]:
epochs = 2000

print("--- ENTRAÎNEMENT DU PROBLÈME INVERSE AVEC ADAM ---")
for epoch in range(epochs):
    optimizer.zero_grad()

    loss_iv = calc_iv_loss(modele, t_init, x_init, u_exact_init)
    loss_bc = calc_bc_loss(modele, t_bord, x_bord, u_exact_bord)
    loss_clp = calc_clp_loss(modele, t_colloc, x_colloc, modele.alpha)
    loss_data = calc_data_loss(modele, inputs_data, u_data)

    loss_totale = 300 * loss_iv + 100 * loss_bc + loss_clp + loss_data
    loss_totale.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoque {epoch:05d} | "
              f"Loss totale: {loss_totale.item():.2e} | "
              f"CLP: {loss_clp.item():.2e} | "
              f"BC: {loss_bc.item():.2e} | "
              f"IV: {loss_iv.item():.2e} | "
              f"Alpha estimé : {modele.alpha.item():.2e}")

## 6. Visualizing the Results
Comparison of the solution learned by the PINN with the exact analytical solution of the heat equation.

In [ ]:
import sys; sys.path.append("..")
import numpy as np                                                                                                    
from pinnplot import plot_solution                        

u_exact = lambda t, x: np.sin(2*np.pi*x) * np.exp(-(2*np.pi)**2 * alpha_vrai * t)

plot_solution(modele, u_exact)

## Comparaison alternative (Adam + SGD séparés)
Cellule alternative de comparaison utilisant un Adam (Réseau) et un SGD (Alpha) séparés avec gradient clipping pour éviter les NaN au démarrage.

In [ ]:
# Re-instanciation propre
modele_sol2 = PinnHeatEq().to(device)

# Deux optimiseurs distincts
optimizer_net = optim.Adam([p for n, p in modele_sol2.named_parameters() if n != 'alpha'], lr=0.001)
optimizer_alpha = optim.SGD([modele_sol2.alpha], lr=0.005) # SGD avec lr modéré pour alpha

epochs = 2000
print("--- DÉBUT ENTRAÎNEMENT SOLUTION 2 (Adam + SGD séparés) ---")
for epoch in range(epochs):
    optimizer_net.zero_grad()
    optimizer_alpha.zero_grad()
    
    loss_iv = calc_iv_loss(modele_sol2, t_init, x_init, u_exact_init)
    loss_bc = calc_bc_loss(modele_sol2, t_bord, x_bord, u_exact_bord)
    loss_clp = calc_clp_loss(modele_sol2, t_colloc, x_colloc, modele_sol2.alpha)
    loss_data = calc_data_loss(modele_sol2, inputs_data, u_data)
    
    loss_totale = 300 * loss_iv + 100 * loss_bc + loss_clp + loss_data
    loss_totale.backward()
    
    # Gradient clipping in case of diverging
    torch.nn.utils.clip_grad_value_([modele_sol2.alpha], clip_value=0.1)
    
    optimizer_net.step()
    optimizer_alpha.step()
    
    if epoch % 100 == 0:
        print(f"Epoque {epoch:05d} | Loss totale: {loss_totale.item():.2e} | CLP: {loss_clp.item():.2e} | Alpha estimé : {modele_sol2.alpha.item():.2e}")